# Part 2D — Final Evaluation: All Models Compared
**Unit:** CIS143-6 Applications of AI  
> Run after all previous notebooks. Loads all saved models from Google Drive.
> Produces: comparison table, precision-recall curves, confusion matrices, bar chart, error analysis.

## 0. Setup & Load All Models

In [ ]:
import os, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print('TensorFlow:', tf.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/brain_tumour_mri')
DATA_DIR = Path('/content/brain_tumour_data')

if not DATA_DIR.exists():
    shutil.copytree(DRIVE_DIR / 'dataset', DATA_DIR)

TRAIN_DIR = DATA_DIR / 'Training'
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
records = []
for cls in CLASSES:
    for img_path in (TRAIN_DIR / cls).glob('*.jpg'):
        records.append({'filepath': str(img_path), 'label': cls})

df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

# Standard test generator
val_test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

CLASS_NAMES = list(test_gen.class_indices.keys())
y_true = test_gen.classes
y_true_bin = label_binarize(y_true, classes=list(range(4)))

# Segmented test generator (for CNN Exp B)
def otsu_rescale(img_array):
    img = img_array / 255.0
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        clean_mask = np.zeros_like(mask)
        cv2.drawContours(clean_mask, [largest], -1, 255, -1)
        mask = clean_mask
    return img * np.stack([mask / 255.0] * 3, axis=-1).astype(np.float32)

seg_test_datagen = ImageDataGenerator(preprocessing_function=otsu_rescale)
seg_test_gen = seg_test_datagen.flow_from_dataframe(
    test_df, x_col='filepath', y_col='label', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

print(f'Test set: {len(test_df)} images  |  Classes: {CLASS_NAMES}')

In [ ]:
model_files = {
    'Baseline CNN':          'baseline_cnn.keras',
    'ANN':                   'ann_model.keras',
    'CNN Exp A':             'cnn_expA_best.keras',
    'CNN Exp B (Otsu)':      'cnn_expB_best.keras',
    'EfficientNetB0 Exp A':  'effnet_frozen.keras',
    'EfficientNetB0 Exp B':  'effnet_finetuned.keras',
}

models_loaded = {}
for name, fname in model_files.items():
    path = DRIVE_DIR / fname
    if path.exists():
        models_loaded[name] = tf.keras.models.load_model(str(path))
        print(f'Loaded: {name}')
    else:
        print(f'MISSING: {name} ({path})')

In [ ]:
# Collect predictions for all models
all_probs = {}
all_preds = {}

for name, model in models_loaded.items():
    gen = seg_test_gen if 'Otsu' in name else test_gen
    gen.reset()
    probs = model.predict(gen, verbose=0)
    all_probs[name] = probs
    all_preds[name] = np.argmax(probs, axis=1)

print('Predictions collected for all models.')

## 1. Final Comparison Table

In [ ]:
results = {}
for name, preds in all_preds.items():
    true = y_true  # all models evaluated on same test set
    rep = classification_report(true, preds, output_dict=True)
    gen = seg_test_gen if 'Otsu' in name else test_gen
    gen.reset()
    loss, acc = models_loaded[name].evaluate(gen, verbose=0)
    results[name] = {
        'Accuracy':       round(acc, 4),
        'Macro Precision':round(rep['macro avg']['precision'], 4),
        'Macro Recall':   round(rep['macro avg']['recall'], 4),
        'Macro F1':       round(rep['macro avg']['f1-score'], 4),
    }

results_df = pd.DataFrame(results).T
print('=== FINAL MODEL COMPARISON ===')
print(results_df.to_string())
results_df.style.highlight_max(color='lightgreen', axis=0)

## 2. Precision-Recall Curves — All Models

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
colours_cls = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for ax, (model_name, probs) in zip(axes.flat, all_probs.items()):
    micro_ap_vals = []
    for i, cls in enumerate(CLASS_NAMES):
        p, r, _ = precision_recall_curve(y_true_bin[:, i], probs[:, i])
        ap = average_precision_score(y_true_bin[:, i], probs[:, i])
        micro_ap_vals.append(ap)
        ax.plot(r, p, label=f'{cls} (AP={ap:.2f})', color=colours_cls[i], lw=2)
    ax.set_title(f'{model_name}\nmicro-avg AP={np.mean(micro_ap_vals):.2f}', fontsize=9)
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.05)

plt.suptitle('Precision-Recall Curves — All Models', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Side-by-Side Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax, (model_name, preds) in zip(axes.flat, all_preds.items()):
    cm = confusion_matrix(y_true, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    ax.set_title(model_name, fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Confusion Matrices — All Models', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Normalised confusion matrix for best model
best_name = results_df['Macro F1'].idxmax()
best_preds = all_preds[best_name]
cm_best = confusion_matrix(y_true, best_preds)
cm_norm = cm_best.astype(float) / cm_best.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
ConfusionMatrixDisplay(cm_best, display_labels=CLASS_NAMES).plot(ax=ax1, cmap='Blues', values_format='d')
ax1.set_title(f'{best_name} — Raw Counts')

ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_NAMES).plot(ax=ax2, cmap='Blues', values_format='.2f')
ax2.set_title(f'{best_name} — Normalised (Proportion)')

plt.suptitle(f'Best Model: {best_name}', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Error Analysis — Best Model

In [ ]:
from PIL import Image as PILImage

best_gen = seg_test_gen if 'Otsu' in best_name else test_gen
best_gen.reset()
best_probs = all_probs[best_name]
best_preds_arr = all_preds[best_name]
filepaths = best_gen.filepaths

misclassified = [
    (filepaths[i], y_true[i], best_preds_arr[i], best_probs[i].max())
    for i in range(len(y_true))
    if y_true[i] != best_preds_arr[i]
]
print(f'Misclassified by {best_name}: {len(misclassified)} / {len(y_true)} '
      f'({len(misclassified)/len(y_true)*100:.1f}%)')

# Confusion pair frequency
pair_counts = {}
for _, t, p, _ in misclassified:
    key = f'{CLASS_NAMES[t]} → {CLASS_NAMES[p]}'
    pair_counts[key] = pair_counts.get(key, 0) + 1
print('\nMost common confusion pairs:')
for pair, cnt in sorted(pair_counts.items(), key=lambda x: -x[1])[:8]:
    print(f'  {pair}: {cnt}')

In [ ]:
samples = random.sample(misclassified, min(6, len(misclassified)))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, (path, true_idx, pred_idx, conf) in zip(axes.flat, samples):
    img = PILImage.open(path).resize((224, 224))
    ax.imshow(img, cmap='gray')
    ax.set_title(
        f'True:  {CLASS_NAMES[true_idx]}\nPred: {CLASS_NAMES[pred_idx]}  ({conf:.1%})',
        fontsize=9, color='firebrick', fontweight='bold'
    )
    ax.axis('off')

plt.suptitle(f'Misclassified Samples — {best_name}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Macro F1 Comparison Bar Chart

In [ ]:
sorted_df = results_df.sort_values('Macro F1')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(sorted_df.index, sorted_df['Macro F1'],
               color=plt.cm.RdYlGn(sorted_df['Macro F1'].values),
               edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=10)
ax.set_xlim(0, 1.1)
ax.set_xlabel('Macro F1 Score', fontsize=12)
ax.set_title('Model Comparison — Macro F1 Score', fontsize=13, pad=12)
ax.axvline(0.5, color='grey', linestyle='--', alpha=0.5, label='0.5 baseline')
ax.legend(fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()

print(f'\nBest model: {best_name}')
print(f'Best Macro F1: {results_df.loc[best_name, "Macro F1"]}')
print(f'Best Accuracy: {results_df.loc[best_name, "Accuracy"]}')

## 6. Per-Class F1 Heatmap — All Models

In [ ]:
perclass_f1 = {}
for name, preds in all_preds.items():
    rep = classification_report(y_true, preds, target_names=CLASS_NAMES, output_dict=True)
    perclass_f1[name] = {cls: rep[cls]['f1-score'] for cls in CLASS_NAMES}

pc_df = pd.DataFrame(perclass_f1).T
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(pc_df, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'F1 Score'})
ax.set_title('Per-Class F1 Score — All Models', fontsize=13, pad=12)
ax.set_xlabel('Class')
ax.set_ylabel('Model')
plt.tight_layout()
plt.show()